# GCN for MDM2 Inhibitor Classification (Google Colab)

Binary classifier that predicts whether a molecule inhibits **MDM2** (target **CHEMBL5023**) from its SMILES string, using a **Graph Convolutional Network (GCN)** built with PyTorch Geometric.

**Paper being replicated:** *Machine Learning-Guided Discovery of Natural MDM2 Inhibitors: A Multistage In Silico Pipeline from Screening to ADMET Profiling* (Budha et al., Advanced Theory and Simulations 2026, DOI [10.1002/adts.202501502](https://doi.org/10.1002/adts.202501502)).

**What we replace:** the paper used a RandomForestClassifier on 2D descriptors for the screening/classification step. We substitute a GCN that operates directly on molecular graphs (atoms = nodes, bonds = edges), which captures topology that 2D fingerprints can miss. Everything downstream in the paper (docking, MD, DFT, ADMET) is unchanged and out of scope here.

**Task & label:** active (MDM2 inhibitor) iff `pChEMBL >= 6.0` (default threshold; ~1 uM potency), else inactive.

**Pipeline in this notebook:**

1. Environment check (GPU/CPU)
2. Install dependencies
3. Get the repo (clone or upload a zip)
4. Download ChEMBL MDM2 bioactivity data
5. Train the GCN
6. Evaluate on the held-out test split
7. Predict on a few example SMILES

Run the cells top to bottom. The whole run takes ~15-30 min on the free CPU runtime and ~5-10 min with a GPU runtime enabled (**Runtime > Change runtime type > GPU**).

In [ ]:
# --- Environment check ---
import platform, sys
print("Python:", platform.python_version())
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: none (CPU runtime) -> training will use the CPU config")
try:
    import torch_geometric
    print("PyG:", torch_geometric.__version__)
except ImportError:
    print("PyG: not installed yet (next cell installs it)")
try:
    from rdkit import Chem
    print("RDKit:", Chem.rdBase.rdkitVersion)
except ImportError:
    print("RDKit: not installed yet")

In [ ]:
%%capture
# --- Install dependencies ---
# Colab already ships torch; installing torch_geometric pulls the right deps for it.
!pip install -q rdkit torch_geometric pandas numpy scikit-learn matplotlib seaborn requests tqdm pyyaml
# On a CPU-only runtime this installs (or keeps) a CPU torch wheel; on a GPU runtime
# it installs the CUDA-enabled wheel automatically.
!pip install -q torch

In [ ]:
# --- Verify install ---
import torch_geometric, rdkit, sklearn, pandas, numpy
print("PyG", torch_geometric.__version__, "| RDKit", rdkit.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# --- Get the repo (clone OR force-sync to the latest GitHub code) ---
# This cell makes the runtime always match GitHub, so stale/broken code in
# an old session never survives a re-run (fixes the previous
# 'unrecognized arguments: --inproc' and 'unable to mmap' errors).
import os, subprocess, shutil

REPO_URL = "https://github.com/Techbjd/gcn-code.git"
ROOT = "/content/gcn-code"

# Remove a stale nested clone left by earlier uploads, if present.
nested = os.path.join(ROOT, "gcn-code")
if os.path.isdir(nested):
    shutil.rmtree(nested)

if os.path.isdir(ROOT) and os.path.exists(os.path.join(ROOT, "src")):
    print("gcn-code present -> force-syncing to latest GitHub code...")
    subprocess.run(["git", "-C", ROOT, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "origin/main"], check=True)
    print("Synced to latest.")
else:
    subprocess.run(["git", "clone", REPO_URL, ROOT], check=True)
    print("Cloned.")

In [ ]:
# --- cd into the repo root ---
import os
if os.path.isdir("/content/gcn-code") and os.path.exists("/content/gcn-code/src"):
    %cd /content/gcn-code
elif os.path.exists("/content/src"):
    %cd /content
else:
    print("WARNING: repo root not found. Re-run the clone/upload cell (zip must contain src/).")
%pwd
!ls -la

In [ ]:
# --- Download ChEMBL MDM2 (CHEMBL5023) bioactivity data ---
# Skips if data/raw/chembl_mdm2.csv already exists (cached).
import os
os.makedirs("data/raw", exist_ok=True)
!python -m src.data.download_chembl --config config/cpu.yaml
print("---") if os.path.exists("data/raw/chembl_mdm2.csv") else print("no CSV yet")
!head -3 data/raw/chembl_mdm2.csv 2>/dev/null || true

In [ ]:
# --- Train the GCN with the right device config ---
# GPU detected -> config/colab.yaml is a copy of gpu.yaml (device: cuda, with CPU
# fallback handled in src/utils.get_device). CPU runtime -> a copy of cpu.yaml.
import os, shutil, torch
if torch.cuda.is_available():
    shutil.copy("config/gpu.yaml", "config/colab.yaml")
    print("GPU detected -> config/colab.yaml (device: cuda)")
else:
    shutil.copy("config/cpu.yaml", "config/colab.yaml")
    print("CPU runtime -> config/colab.yaml (device: cpu)")
!python -m src.train --config config/colab.yaml

In [ ]:
# --- Evaluate the best checkpoint on the held-out test split ---
!python -m src.evaluate --config config/colab.yaml
print("--- outputs/test_metrics.json ---")
!cat outputs/test_metrics.json

In [ ]:
# --- Predict activity for a few example SMILES ---
# Predictions sorted by probability descending; class=1 if prob >= 0.5.
!python -m src.predict --config config/colab.yaml \
    --smiles_file data/raw/example_smiles.csv \
    --output outputs/predictions_colab.csv
print("--- outputs/predictions_colab.csv ---")
!cat outputs/predictions_colab.csv

In [ ]:
# --- Screen COCONUT with the trained GCN (~738k natural products) ---
# Downloads the COCONUT lite CSV once, then scores it in chunks.
# `--inproc` featurizes in this process: slowest but 100% safe on Colab's
# limited RAM. Drop `--inproc` to use parallel worker processes (faster; the
# workers now return numpy instead of torch tensors, so the old
# 'unable to mmap ... Cannot allocate memory' crash is fixed).
import os, zipfile, glob
if not os.path.exists("coconut_csv_lite.csv"):
    os.system("wget -q -O coconut.zip https://coconut.s3.uni-jena.de/prod/downloads/2026-08/coconut_csv_lite-08-2026.zip")
    with zipfile.ZipFile("coconut.zip") as z:
        z.extractall("coconut_data")
    os.rename(glob.glob("coconut_data/*.csv")[0], "coconut_csv_lite.csv")
    print("COCONUT CSV ready.")
!python -m src.screen --config config/colab.yaml \
    --input coconut_csv_lite.csv \
    --smiles_col canonical_smiles --id_col id \
    --output outputs/coconut_predictions.csv \
    --chunk_size 50000 --inproc

In [ ]:
# --- Summarize hits + build a diverse docking set ---
!python -m src.analyze_hits --predictions outputs/coconut_predictions.csv \
    --library coconut_csv_lite.csv \
    --id_col id --smiles_col canonical_smiles \
    --n_diverse 200 --outdir outputs

# Results & Next Steps

**What you should see:**

- Per-epoch training lines with train loss/acc and val loss/acc/auc, early stopping on val loss.
- `outputs/test_metrics.json` with accuracy, ROC-AUC, PR-AUC, precision, recall, F1.
- `outputs/plots/{roc_curve,pr_curve,confusion_matrix}.png` and `outputs/training_history.csv`.
- `outputs/predictions_colab.csv` ranking the example molecules by predicted inhibition probability.

**COCONUT screening (large library):**

1. `src.screen` downloads the COCONUT lite CSV once, then scores all ~738k natural products in parallel chunks. Output: `outputs/coconut_predictions.csv` (ranked by `probability_active`).
2. `src.analyze_hits` summarizes the screen, dedups structural duplicates, and keeps one molecule per Murcko scaffold to produce a diverse docking set: `outputs/hits.csv` (all hits) and `outputs/diverse_hits.csv` / `.smi` (diverse set, ready for docking).

**Next steps (following the paper's downstream stages):**

1. **Docking** the diverse hits against MDM2 (e.g., AutoDock Vina/Glide) to filter by binding pose and score.
2. **Molecular dynamics (MD)** on the best complexes to confirm stability.
3. **DFT / ADMET profiling** of the surviving candidates.

**Interpretation caveats:** the GCN is a screening prior, not a docking oracle — a high score means the model believes the molecule's graph resembles known MDM2 inhibitors, not that it binds. Many COCONUT molecules score exactly 1.0 (probability saturation); do not rely on ordering within that tier — the scaffold-diverse set is the right thing to dock. Validate computationally (docking/MD) and experimentally before drawing conclusions. Research use only.